# Week 4 - Feature Selection + PCA

**Goal:** Comparing 4 dimensionality-reduction / feature-selection approaches: Sequential Forward Selection (SFS), Sequential Backward Selection (SBS), Bidirectional Selection, and PCA

## 0. Setup

In [34]:
import os
import sys
import platform
import json
import math
import random
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import recall_score, confusion_matrix
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.model_selection import cross_val_score

import numpy as np
import pandas as pd

from matplotlib import pyplot as plt

print('Python:', sys.version.split()[0])
print('Platform:', platform.platform())
print('pandas:', pd.__version__)

Python: 3.13.1
Platform: Windows-10-10.0.19045-SP0
pandas: 3.0.1


In [ ]:
from pathlib import Path

PROJECT_ROOT = Path.cwd() / "no_show_predict"
DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_INTERIM = PROJECT_ROOT / "data" / "interim"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
REPORTS = PROJECT_ROOT / "reports"
CONFIGS = PROJECT_ROOT / "configs"

PROJECT_ROOT

WindowsPath('c:/Users/Vivi/Documents/Repos/Data-Science/Week-4')

## A. Setup + Baseline

1. Load dataset & define:
    - `x` = feature matrix (numeric features only; encode categories if needed)
    - `y` = target

In [4]:
data_path = DATA_PROCESSED / "appointments_processed.csv"
data = pd.read_csv(data_path)
print(data.shape)
data.head()

(110527, 15)


,PatientId,AppointmentID,Gender,ScheduledDay,AppointmentDay,Age,Neighbourhood,Scholarship,Hypertension,Diabetes,Alcoholism,Handicap,SMSReceived,NoShow,WaitingDays
0,2.987250e+13,5642903,F,2016-04-29 18:38:08+00:00,2016-04-29 00:00:00+00:00,62,JARDIM DA PENHA,False,True,False,False,False,False,False,0
1,5.589978e+14,5642503,M,2016-04-29 16:08:27+00:00,2016-04-29 00:00:00+00:00,56,JARDIM DA PENHA,False,False,False,False,False,False,False,0
2,4.262962e+12,5642549,F,2016-04-29 16:19:04+00:00,2016-04-29 00:00:00+00:00,62,MATA DA PRAIA,False,False,False,False,False,False,False,0
3,8.679512e+11,5642828,F,2016-04-29 17:29:31+00:00,2016-04-29 00:00:00+00:00,8,PONTAL DE CAMBURI,False,False,False,False,False,False,False,0
4,8.841186e+12,5642494,F,2016-04-29 16:07:23+00:00,2016-04-29 00:00:00+00:00,56,JARDIM DA PENHA,False,True,True,False,False,False,False,0


In [27]:
# Target
y = data["NoShow"]

# Features
x = (
    data
    .drop(columns = ["NoShow"])
    .select_dtypes(include = ["number", "bool"])
)

# Train/test split
x_train, x_test, y_train, y_test = train_test_split(
    x,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Number of features:", x.shape[1])
print("Feature names:")
for col in x.columns:
    print("-", col)
print()

print("Train/Test split:")
print("- x_train:", x_train.shape)
print("- x_test:", x_test.shape)
print("- y_train:", y_train.shape)
print("- y_test:", y_test.shape)
print()

metric_name = "Recall"
print("Chosen metric:", metric_name)

Number of features: 10
Feature names:
- PatientId
- AppointmentID
- Age
- Scholarship
- Hypertension
- Diabetes
- Alcoholism
- Handicap
- SMSReceived
- WaitingDays

Train/Test split:
- x_train: (88421, 10)
- x_test: (22106, 10)
- y_train: (88421,)
- y_test: (22106,)

Chosen metric: Recall


2. Create baseline model using all features
    - Classification: Logistic Regression

In [28]:
baseline_model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        max_iter = 1000,
        random_state = 42
    ))
])

print("Task: Classification")
print("Model:", baseline_model.named_steps["model"])

Task: Classification
Model: LogisticRegression(max_iter=1000, random_state=42)


3. Evaluate baseline with your chosen metric:
    - Classification: recall

In [30]:
baseline_model.fit(x_train, y_train)
y_pred = baseline_model.predict(x_test)
baseline_recall = recall_score(y_test, y_pred, pos_label = 1)

print("Baseline Recall Score:", baseline_recall)
print("Confusion Matrix (TP, FN, FP, TN):")
print(confusion_matrix(y_test, y_pred))

Baseline Recall Score: 0.012544802867383513
Confusion Matrix (TP, FN, FP, TN):
[[17521   121]
 [ 4408    56]]


4. Why choose this metric for this problem?
    
    Because recall measures how well the model captures the positive class.
    The main goal is to detect as many actual no-show patients as possible so the hospital can intervene. Incorrectly flagging a patient as a possible no-show is less costly than missing a correct one.

## B. Sequential Feature Selection

### B1. Sequential Forward Selection (SFS)

1. Run SFS to select k features (choose k = 5 OR k = round(0.25 × number_of_features)).

In [33]:
nr_features = x_train.shape[1]
# k = round(0.25 * nr_features)
k = 5

print("Number of original features:", nr_features)
print("Number of features to select", k)

Number of original features: 10
Number of features to select 5


In [36]:
sfs_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("sfs", SequentialFeatureSelector(
        LogisticRegression(
            max_iter = 1000,
            random_state = 42
        ),
        n_features_to_select = k,
        direction = "forward",
        scoring = "recall",
        cv = 5,
        n_jobs = -1
    )),
    ("model", LogisticRegression(
        max_iter = 1000,
        random_state = 42
    ))
])

sfs_pipeline.fit(x_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('scaler', ...), ('sfs', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True
,estimator estimator: estimator instanceAn unfitted estimator.,LogisticRegre...ndom_state=42)
,"n_features_to_select n_features_to_select: ""auto"", int or float, default=""auto""If `""auto""`, the behaviour depends on the `tol` parameter:- if `tol` is not `None`, then features are selected while the score change does not exceed `tol`.- otherwise, half of the features are selected.If integer, the parameter is the absolute number of features to select.If float between 0 and 1, it is the fraction of features to select... versionadded:: 1.1 The option `""auto""` was added in version 1.1... versionchanged:: 1.3 The default changed from `""warn""` to `""auto""` in 1.3.",5
,"tol tol: float, default=NoneIf the score is not incremented by at least `tol` between twoconsecutive feature additions or removals, stop adding or removing.`tol` can be negative when removing features using `direction=""backward""`.`tol` is required to be strictly positive when doing forward selection.It can be useful to reduce the number of features at the cost of a smalldecrease in the score.`tol` is enabled only when `n_features_to_select` is `""auto""`... versionadded:: 1.1",None
,"direction direction: {'forward', 'backward'}, default='forward'Whether to perform forward selection or backward selection.",'forward'


2. Report:
    - Selected feature list
    - CV score during selection
    - Test-set score using only the selected features

In [38]:
# Selected feature list
selected_mask = sfs_pipeline.named_steps["sfs"].get_support()
selected_features = x_train.columns[selected_mask].tolist()

# CV score during selection
x_train_selected = x_train[selected_features]
cv_scores = cross_val_score(
    Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(
            max_iter = 1000,
            random_state = 42
        ))
    ]),
    x_train_selected,
    y_train,
    cv = 5,
    scoring = "recall",
    n_jobs = -1
)

# Test score using only selected fatures
x_test_selected = x_test[selected_features]
final_model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        max_iter = 1000,
        random_state = 42
    ))
])
final_model.fit(x_train_selected, y_train)
y_pred_selected = final_model.predict(x_test_selected)
test_recall_selected = recall_score(y_test, y_pred_selected, pos_label = 1)

# Print report
print("SFS Report:")
print("- Number of selected features:", len(selected_features))
print("- Selected features:")
for feature in selected_features:
    print("\t -", feature)
print("- Mean CV recall:", cv_scores.mean())
print("- Test recall with selected features:", test_recall_selected)

SFS Report:
- Number of selected features: 5
- Selected features:
	 - AppointmentID
	 - Age
	 - Diabetes
	 - Alcoholism
	 - WaitingDays
- Mean CV recall: 0.020442453094371326
- Test recall with selected features: 0.01904121863799283


### B2. Sequential Backward Selection (SBS)

1. Run SBS to select the same k features.

In [41]:
sbs_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("sbs", SequentialFeatureSelector(
        LogisticRegression(
            max_iter = 1000,
            random_state = 42
        ),
        n_features_to_select = k,
        direction = "backward",
        scoring = "recall",
        cv = 5,
        n_jobs = -1
    )),
    ("model", LogisticRegression(
        max_iter = 1000,
        random_state = 42
    ))
])

sbs_pipeline.fit(x_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('scaler', ...), ('sbs', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True
,estimator estimator: estimator instanceAn unfitted estimator.,LogisticRegre...ndom_state=42)
,"n_features_to_select n_features_to_select: ""auto"", int or float, default=""auto""If `""auto""`, the behaviour depends on the `tol` parameter:- if `tol` is not `None`, then features are selected while the score change does not exceed `tol`.- otherwise, half of the features are selected.If integer, the parameter is the absolute number of features to select.If float between 0 and 1, it is the fraction of features to select... versionadded:: 1.1 The option `""auto""` was added in version 1.1... versionchanged:: 1.3 The default changed from `""warn""` to `""auto""` in 1.3.",5
,"tol tol: float, default=NoneIf the score is not incremented by at least `tol` between twoconsecutive feature additions or removals, stop adding or removing.`tol` can be negative when removing features using `direction=""backward""`.`tol` is required to be strictly positive when doing forward selection.It can be useful to reduce the number of features at the cost of a smalldecrease in the score.`tol` is enabled only when `n_features_to_select` is `""auto""`... versionadded:: 1.1",None
,"direction direction: {'forward', 'backward'}, default='forward'Whether to perform forward selection or backward selection.",'backward'


2. Report:
    - Selected feature list
    - CV score during selection
    - Test-set score using only the selected features

In [42]:
# Selected feature list
selected_mask = sbs_pipeline.named_steps["sbs"].get_support()
selected_features = x_train.columns[selected_mask].tolist()

# CV score during selection
x_train_selected = x_train[selected_features]
cv_scores = cross_val_score(
    Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(
            max_iter = 1000,
            random_state = 42
        ))
    ]),
    x_train_selected,
    y_train,
    cv = 5,
    scoring = "recall",
    n_jobs = -1
)

# Test score using only selected fatures
x_test_selected = x_test[selected_features]
final_model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        max_iter = 1000,
        random_state = 42
    ))
])
final_model.fit(x_train_selected, y_train)
y_pred_selected = final_model.predict(x_test_selected)
test_recall_selected = recall_score(y_test, y_pred_selected, pos_label = 1)

# Print report
print("SBS Report:")
print("- Number of selected features:", len(selected_features))
print("- Selected features:")
for feature in selected_features:
    print("\t -", feature)
print("- Mean CV recall:", cv_scores.mean())
print("- Test recall with selected features:", test_recall_selected)

SBS Report:
- Number of selected features: 5
- Selected features:
	 - AppointmentID
	 - Age
	 - Diabetes
	 - Alcoholism
	 - WaitingDays
- Mean CV recall: 0.020442453094371326
- Test recall with selected features: 0.01904121863799283


3. Did SBS pick a very different set than SFS? Why?

    They picked the same set of features, indicating that the most relevant predictors are relatively stable. Another contributor is that there are not many original features to begin with (10).

### B3. Bidirectional Selection

1. Run bidirectional selection (forward + backward steps) to select the same k features.

2. Report:
    - Selected feature list
    - CV score during selection
    - Test-set score using only the selected features

3. When might bidirectional selection be preferable to pure forward/backward?

## C. PCA as Dimensionality Reduction

### C1. Fit PCA (with scaling)

1. Build a pipeline
    - Imputer (if needed)
    - Scaler (standardization)
    - PCA
    - Estimator (same model family as baseline if possible)

2. Plot the cumulative explained variance vs number of components.

3. Choose number of components using one rule:
    - Minimum components to reach 90% explained variance, OR
    - Use an “elbow” point you justify briefly

### C2. Evaluate PCA Model

1. Train and evaluate the model using your chosen number of PCA components.

2. Report:
    - Number of components used
    - Explained variance achieved
    - CV score (optional but recommended)
    - Test-set score

3. Benefits & drawbacks of PCA compared to selecting original features:
    - PRO:
    - CON: